# CalCOFI DIC 予測(MLP)

方針
- 入力は元の列(深さ・栄養塩は対数)+ 観測地点の one-hot。
- tanh の小さな MLP を L-BFGS で全データ一括学習し、外れ値に強い損失(pseudo-Huber)を使う。
- 設定と種を変えた 18 本の MLP の予測を平均して提出する。

In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import math, random, time
from sklearn.model_selection import KFold

# Load data
DATA = "/kaggle/input/competitions/eds-232-ocean-chemistry-prediction-for-calcofi"
if not os.path.exists(DATA):
    DATA = "data"
train = pd.read_csv(f"{DATA}/train.csv")
test = pd.read_csv(f"{DATA}/test.csv")
train = train.rename(columns={"TA1.x": "TA1"})
y = train["DIC"].values.astype(np.float64)
print(train.shape, test.shape)

# ---- 入力 ----
COLS = {
    "log": ["Lat_Dec", "Lon_Dec", "NO2uM", "NO3uM", "NH3uM", "R_Depth", "R_Sal", "R_DYNHT",
            "R_Oxy_micromol.Kg", "PO4uM", "SiO3uM", "TA1", "Salinity1", "Temperature_degC"],
    "geolog": ["Lat_Dec", "Lon_Dec", "NO3uM", "R_Oxy_micromol.Kg", "PO4uM", "SiO3uM",
               "TA1", "Salinity1", "Temperature_degC", "R_Depth"],
}
LOG_COLS = ["R_Depth", "NO2uM", "NH3uM", "SiO3uM"]

def features(df, mode):
    x = df[COLS[mode]].astype(np.float64).copy()
    # 深さ・栄養塩は対数にする
    for c in LOG_COLS:
        if c in x:
            x[c] = np.log1p(np.maximum(0, x[c]))
    return x.values

X = {m: features(train, m) for m in COLS}
X_test = {m: features(test, m) for m in COLS}

In [ ]:
class MLPModel:
    """tanh の順伝播 MLP。L-BFGS で学習する。"""
    def __init__(self, input_size, hidden, wd, seed):
        rng = random.Random(seed)
        dims = [input_size] + hidden + [1]
        parts, self.shapes = [], []
        for i, (a, b) in enumerate(zip(dims[:-1], dims[1:])):
            scale = (0.1 if i == len(dims) - 2 else 1) / math.sqrt(a)
            parts += [np.array([rng.gauss(0, scale) for _ in range(a * b)]), np.zeros(b)]
            self.shapes += [(a, b), (b,)]
        self.w = np.concatenate(parts)
        self.wd = wd

    def unpack(self, w):
        out, k = [], 0
        for s in self.shapes:
            n = math.prod(s)
            out.append(w[k:k + n].reshape(s))
            k += n
        return out

    def forward(self, w, x):
        ps = self.unpack(w)
        acts = [x]
        for i in range(0, len(ps), 2):
            z = acts[-1] @ ps[i] + ps[i + 1]
            acts.append(z if i == len(ps) - 2 else np.tanh(z))
        return acts, ps

    def loss_grad(self, w, x, y, delta, weights):
        acts, ps = self.forward(w, x)
        e = acts[-1][:, 0] - y
        # pseudo-Huber 損失
        a = np.sqrt(1 + (e / delta) ** 2)
        losses, grad = 2 * delta * delta * (a - 1), 2 * e / a
        if weights is not None:
            losses, grad = losses * weights, grad * weights
        loss = float(losses.mean() + self.wd * (w @ w))
        back = grad[:, None] / len(y)
        g = np.empty_like(w)
        gp = self.unpack(g)
        for layer in range(len(acts) - 2, -1, -1):
            gp[2 * layer][:] = acts[layer].T @ back
            gp[2 * layer + 1][:] = back.sum(axis=0)
            if layer:
                back = (back @ ps[2 * layer].T) * (1 - acts[layer] ** 2)
        g += 2 * self.wd * w
        return loss, g

    def fit(self, x, y, delta, weights=None, iters=1200):
        # L-BFGS(履歴12本)+ 直線探索
        w = self.w
        f, g = self.loss_grad(w, x, y, delta, weights)
        hist = []
        for _ in range(iters):
            q, alphas = g.copy(), []
            for s, v, rho in reversed(hist):
                a = rho * (s @ q)
                alphas.append(a)
                q -= a * v
            d = q * ((hist[-1][0] @ hist[-1][1]) / (hist[-1][1] @ hist[-1][1]) if hist else 1)
            for (s, v, rho), a in zip(hist, reversed(alphas)):
                d += s * (a - rho * (v @ d))
            d = -d
            slope = g @ d
            if slope >= 0:
                hist, d, slope = [], -g, -(g @ g)
            rate = 1.0
            for _ in range(24):
                wn = w + rate * d
                fn, gn = self.loss_grad(wn, x, y, delta, weights)
                if np.isfinite(fn) and fn <= f + 1e-4 * rate * slope:
                    break
                rate *= 0.5
            else:
                break
            s, v = wn - w, gn - g
            if s @ v > 1e-14:
                hist.append((s, v, 1 / (s @ v)))
                if len(hist) > 12:
                    hist.pop(0)
            w, f, g = wn, fn, gn
        self.w = w
        return self

    def predict(self, x):
        return self.forward(self.w, x)[0][-1][:, 0]

In [ ]:
def fit_predict(x, y, v, cfg):
    """学習側 x, y で学習し、v の予測を返す。標準化と地点の列は学習側だけで作る。"""
    mean, std = x.mean(axis=0), np.maximum(x.std(axis=0), 1e-8)
    xx = np.clip((x - mean) / std, -8, 8)
    vv = np.clip((v - mean) / std, -8, 8)
    if cfg.get("station"):
        # 観測地点(緯度経度を丸めたもの)の one-hot。学習側に2回以上出る地点だけ
        dec = cfg["station_decimals"]
        keys = [tuple(a) for a in np.round(x[:, :2], dec)]
        vkeys = [tuple(a) for a in np.round(v[:, :2], dec)]
        counts = {}
        for k in keys:
            counts[k] = counts.get(k, 0) + 1
        mapping = {k: i for i, k in enumerate(k for k, c in counts.items() if c >= 2)}
        one = np.zeros((len(x), len(mapping)))
        vone = np.zeros((len(v), len(mapping)))
        for i, k in enumerate(keys):
            if k in mapping:
                one[i, mapping[k]] = cfg["station_scale"]
        for i, k in enumerate(vkeys):
            if k in mapping:
                vone[i, mapping[k]] = cfg["station_scale"]
        xx, vv = np.column_stack((xx, one)), np.column_stack((vv, vone))
    ym, ys = y.mean(), y.std()
    model = MLPModel(xx.shape[1], cfg["hidden"], cfg["wd"], cfg["seed"])
    model.fit(xx, (y - ym) / ys, cfg["delta"] / ys)
    return model.predict(vv) * ys + ym

# 18本の設定(6グループ × 種3つ)
GROUPS = [
    dict(hidden=[24, 12], wd=3e-5, mode="log", delta=2),
    dict(hidden=[24, 12], wd=3e-5, mode="geolog", delta=8),
    dict(hidden=[24, 12], wd=3e-5, mode="log", delta=4, station=True, station_decimals=3, station_scale=0.25),
    dict(hidden=[24, 12], wd=3e-5, mode="log", delta=4, station=True, station_decimals=4, station_scale=0.25),
    dict(hidden=[12, 6], wd=1e-5, mode="geolog", delta=4, station=True, station_decimals=4, station_scale=0.2),
    dict(hidden=[48, 24], wd=1e-5, mode="geolog", delta=4, station=True, station_decimals=4, station_scale=0.2),
]
CONFIGS = [dict(g, seed=s) for g in GROUPS for s in (200, 201, 202)]
print(len(CONFIGS), "models")

In [ ]:
RUN_CV = True

if RUN_CV:
    t0 = time.time()
    oof = np.full((2, len(y)), np.nan)
    for rep in range(2):
        for fold, (tr, va) in enumerate(KFold(4, shuffle=True, random_state=100 + rep).split(y)):
            preds = [fit_predict(X[c["mode"]][tr], y[tr], X[c["mode"]][va], c) for c in CONFIGS]
            oof[rep, va] = np.mean(preds, axis=0)
            print(f"rep {rep} fold {fold + 1}/4 done ({time.time() - t0:.0f}s)", flush=True)
    rmse_all = np.sqrt(np.mean((oof - y) ** 2))
    print(f"CV RMSE(全行): {rmse_all:.3f}")

## 本番: train 全行で 18 本学習して平均

In [ ]:
test_preds = np.array([fit_predict(X[c["mode"]], y, X_test[c["mode"]], c) for c in CONFIGS])
print(test_preds.shape)

In [ ]:
# Make predictions
predictions = test_preds.mean(axis=0)

# Prepare submission
submission = pd.DataFrame({"id": test["id"].values, "DIC": predictions})
submission.to_csv("submission.csv", index=False)
print(submission.shape, "NaN:", int(submission["DIC"].isna().sum()))
submission.head()

In [ ]:
# score: 3.46799
# name: 
# student_id: 